# GateSense 02: compare OCR engines (EasyOCR vs PaddleOCR)

Run in **Google Colab** (*Runtime → Run all*; a GPU makes it faster but isn't required).

It installs the GateSense ANPR package from GitHub and runs `anpr/eval/evaluate_pipeline.py` with **both** engines on the same labelled images, then compares **plate accuracy** (exact match), **character accuracy** and speed, per condition (day / night / rain / angled / HSRP / two-wheeler).

**What you need:** a folder on Google Drive laid out as in `anpr/DATASET.md`: images plus a `labels.csv` with `file,plate,condition`.
- Full photos (the detector finds the plate): set `IMAGES_ARE_CROPS = False` and have `plate_detector.onnx` from notebook 01 in the same Drive folder.
- Already-cropped plates: set `IMAGES_ARE_CROPS = True` (measures OCR alone).

The last section also runs a **synthetic** sanity set (rendered plates). It is clearly labelled and is **not** used to pick the engine.

In [ ]:
EVAL_DIR = "/content/drive/MyDrive/gatesense/ocr_eval"   # images + labels.csv
IMAGES_ARE_CROPS = False
DETECTOR = "/content/drive/MyDrive/gatesense/plate_detector.onnx"  # used when IMAGES_ARE_CROPS is False
ENGINES = ["easyocr", "paddleocr"]
# Prefer the faster engine unless the other is clearly more accurate (plate accuracy points):
MIN_ACCURACY_GAIN = 0.02

In [ ]:
import sys, subprocess
assert sys.version_info >= (3, 12), f"GateSense needs Python 3.12+, this runtime has {sys.version}"
!git clone -q --depth 1 https://github.com/Akinchan-Jain-7/GateSense /content/GateSense
%pip install -q -e "/content/GateSense/anpr[easyocr,paddleocr]"
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import json
from pathlib import Path

results_dir = Path("/content/ocr_results")
for engine in ENGINES:
    cmd = [sys.executable, "/content/GateSense/anpr/eval/evaluate_pipeline.py", EVAL_DIR,
           "--ocr", engine, "--out", str(results_dir)]
    cmd += ["--crops"] if IMAGES_ARE_CROPS else ["--model", DETECTOR]
    print(" ".join(cmd)); subprocess.run(cmd, check=True)

kind = "ocr" if IMAGES_ARE_CROPS else "e2e"
summary = {e: json.loads((results_dir / f"{kind}_{e}.json").read_text())["summary"] for e in ENGINES}
overall = {e: next(r for r in rows if r["condition"] == "all") for e, rows in summary.items()}
for e, r in overall.items():
    print(f"{e:10s} plate acc {r['plate_accuracy']:.1%}  char acc {r['char_accuracy']:.1%}  {r['mean_latency_ms']:.0f} ms/image")

In [ ]:
a, b = ENGINES
fast, slow = sorted(ENGINES, key=lambda e: overall[e]["mean_latency_ms"])
gain = overall[slow]["plate_accuracy"] - overall[fast]["plate_accuracy"]
chosen = slow if gain >= MIN_ACCURACY_GAIN else fast
reason = (f"{slow} is {gain:.1%} more accurate" if chosen == slow
          else f"{fast} is faster and the other gains < {MIN_ACCURACY_GAIN:.0%} plate accuracy")

conditions = sorted({r["condition"] for rows in summary.values() for r in rows}, key=lambda c: (c == "all", c))
lines = ["| Condition | Images | " + " | ".join(f"{e} plate acc | {e} char acc" for e in ENGINES) + " |",
         "|---|---|" + "---|---|" * len(ENGINES)]
for c in conditions:
    cells = []
    for e in ENGINES:
        r = next(x for x in summary[e] if x["condition"] == c)
        cells += [f"{r['plate_accuracy']:.1%}", f"{r['char_accuracy']:.1%}"]
    n = next(x for x in summary[ENGINES[0]] if x["condition"] == c)["samples"]
    lines.append(f"| {c} | {n} | " + " | ".join(cells) + " |")
speed = " · ".join(f"{e}: {overall[e]['mean_latency_ms']:.0f} ms/image" for e in ENGINES)
report = (f"# OCR engines: {' vs '.join(ENGINES)}\n\n{'Plate crops' if IMAGES_ARE_CROPS else 'Full photos, detector + OCR'}, "
          f"dataset `{EVAL_DIR}`. Colab, {subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout.strip() or 'CPU'}.\n\n"
          + "\n".join(lines) + f"\n\nSpeed: {speed}.\n\n**Chosen:** {chosen} ({reason}).\n")
(results_dir / "ocr_comparison.md").write_text(report)
print(report)

### Optional: synthetic sanity check (not used for the decision)
Renders 200 plates with blur and noise, to check both engines run and to see the typical look-alike mistakes. Rendered fonts are much easier than real plates, so these numbers are **not** comparable to real photos.

In [ ]:
import csv, random, cv2, numpy as np
sys.path.insert(0, "/content/GateSense/anpr")
from gatesense_anpr.simulate import DEFAULT_PLATES, render_plate

synth = Path("/content/synthetic_crops"); synth.mkdir(exist_ok=True)
rng = random.Random(0)
with (synth / "labels.csv").open("w", newline="") as f:
    w = csv.writer(f); w.writerow(["file", "plate", "condition"])
    for i in range(200):
        plate = rng.choice(DEFAULT_PLATES)
        crop = render_plate(plate)[140:220, 120:520]
        condition = rng.choice(["clean", "blurred", "noisy"])
        if condition == "blurred": crop = cv2.GaussianBlur(crop, (7, 7), 0)
        if condition == "noisy": crop = np.clip(crop + np.random.default_rng(i).normal(0, 25, crop.shape), 0, 255).astype(np.uint8)
        cv2.imwrite(str(synth / f"{i:03d}.jpg"), crop); w.writerow([f"{i:03d}.jpg", plate, condition])
for engine in ENGINES:
    subprocess.run([sys.executable, "/content/GateSense/anpr/eval/evaluate_pipeline.py", str(synth),
                    "--ocr", engine, "--crops", "--out", str(results_dir / "synthetic")], check=True)
for engine in ENGINES:
    print((results_dir / "synthetic" / f"ocr_{engine}.md").read_text())

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/gatesense_ocr_results", "zip", results_dir)
files.download("/content/gatesense_ocr_results.zip")

## What to do with the files
Put `ocr_comparison.md` and the `*.json` files in `anpr/eval/results/` (or give them to Claude). The chosen engine becomes the default `ANPR_OCR_ENGINE` and the decision is recorded in ADR-0009.